# 05 Movie Metrics

You've been given a table of Netflix users and another with their viewing activity, including the movie name, date started, and whether they finished it.

Your task is to engineer these new features for each user, based on their activity:

- Date from the first movie they finished
- Name of the first movie they finished
- Date from the last movie they finished
- Name of the last movie they finished
- Movies started
- Movies finished

![image](../images/L6wub14jGNcwG0zLLxlVyIWv85c.avif)

In [1]:
import pandas as pd

In [2]:
users = pd.read_csv('data/users.csv', parse_dates=['created_at'])
users.head()

,id,created_at,country_code
0,1,2023-05-26,CA
1,2,2023-06-15,CA
2,3,2023-07-18,MX
3,4,2023-07-27,CA
4,5,2023-09-01,US


In [3]:

activity = pd.read_csv('data/activity.csv', parse_dates=['date'])
activity.head()


,id,user_id,date,movie_name,finished
0,1,2,2023-06-22,The Shawshank Redemption,1
1,2,1,2023-07-23,Shrek,0
2,3,4,2023-07-27,Fight Club,1
3,4,1,2023-08-23,Top Gun: Maverick,0
4,5,4,2023-08-24,Oppenheimer,0


In [4]:
start_finish = (
    activity
    .groupby('user_id', as_index=False)
    .agg(started_movies = ("finished", "count"),
         finished_movies = ("finished", "sum"))
)

start_finish.head()

,user_id,started_movies,finished_movies
0,1,30,26
1,2,15,12
2,3,10,7
3,4,43,34
4,5,31,25


In [5]:
finished = activity.query('finished == 1')
first_finished = (
    finished
    .sort_values(["user_id","date"])
    .groupby("user_id", as_index=False)
    .agg(first_finished_date = ("date", "first"),
         first_finished_movie = ("movie_name", "first"))
)

first_finished.head()

,user_id,first_finished_date,first_finished_movie
0,1,2023-09-12,Turning Red
1,2,2023-06-22,The Shawshank Redemption
2,3,2023-11-10,Oppenheimer
3,4,2023-07-27,Fight Club
4,5,2023-09-07,Top Gun: Maverick


In [6]:
last_finished = (
    finished
    .sort_values(["user_id","date"], ascending=[True, False])
    .groupby("user_id", as_index=False)
    .agg(last_finished_date = ("date", "first"),
         last_finished_movie = ("movie_name", "first"))
)

last_finished.head()


,user_id,last_finished_date,last_finished_movie
0,1,2025-03-26,Her
1,2,2025-05-01,Fight Club
2,3,2025-03-31,Nope
3,4,2025-05-09,Avengers: Endgame
4,5,2025-03-14,Bohemian Rhapsody


In [7]:
users = (
    users
    .merge(first_finished, how='left', left_on='id', right_on='user_id')
    .merge(last_finished, how='left', left_on='id', right_on='user_id')
    .merge(start_finish, how='left', left_on='id', right_on='user_id')
)

users.head()

,id,created_at,country_code,user_id_x,first_finished_date,first_finished_movie,user_id_y,last_finished_date,last_finished_movie,user_id,started_movies,finished_movies
0,1,2023-05-26,CA,1,2023-09-12,Turning Red,1,2025-03-26,Her,1,30,26
1,2,2023-06-15,CA,2,2023-06-22,The Shawshank Redemption,2,2025-05-01,Fight Club,2,15,12
2,3,2023-07-18,MX,3,2023-11-10,Oppenheimer,3,2025-03-31,Nope,3,10,7
3,4,2023-07-27,CA,4,2023-07-27,Fight Club,4,2025-05-09,Avengers: Endgame,4,43,34
4,5,2023-09-01,US,5,2023-09-07,Top Gun: Maverick,5,2025-03-14,Bohemian Rhapsody,5,31,25


In [9]:
users.query('last_finished_movie=="Fight Club"')

,id,created_at,country_code,user_id_x,first_finished_date,first_finished_movie,user_id_y,last_finished_date,last_finished_movie,user_id,started_movies,finished_movies
1,2,2023-06-15,CA,2,2023-06-22,The Shawshank Redemption,2,2025-05-01,Fight Club,2,15,12
17,18,2024-10-01,CA,18,2024-10-01,Fight Club,18,2025-05-16,Fight Club,18,35,28
19,20,2024-11-23,MX,20,2024-11-27,Luca,20,2025-05-02,Fight Club,20,34,19
